# Chinchilla Scaling Laws – Learning Notes

> A Q&A-style walkthrough building up an intuition for the Chinchilla scaling law, 
> and how LLMs, neural networks, parameters, and tokens all fit together.

## 1. What is the Chinchilla Scaling Law?

**In one sentence:** For a fixed compute budget, the best model comes from *balancing* parameters and training data (about **20 tokens per parameter**) rather than just making the model as big as possible.

Source: DeepMind, 2022 - Hoffmann et al., *"Training Compute-Optimal Large Language Models."*

### The compute budget

$$C \approx 6 \cdot N \cdot D$$

* $N$ = number of model parameters
* $D$ = number of training tokens
* $6$ = approx FLOPs per parameter per token (forward + backward pass)

The same budget $C$ can be spent as *big model + little data* or *small model + lots of data*. Chinchilla asks: **which split gives the lowest loss?**

### The loss formula they fit

$$L(N, D) = E + \frac{A}{N^\alpha} + \frac{B}{D^\beta}$$

* $E$ = irreducible loss (entropy of language itself)
* $A/N^\alpha$ = error from too few **parameters**
* $B/D^\beta$ = error from too little **data**

The fitted exponents were roughly equal ($\alpha \approx \beta \approx 0.34$), so the compute-optimal recipe is to **scale N and D together** – the famous **~20 tokens per parameter** heuristic.


### The headline example

| Model | Parameters | Training Tokens | Result |
| :--- | :--- | :--- | :--- |
| **Gopher** | 280 B | 300 B | baseline |
| **Chinchilla** | **70 B** | **1.4 T** (~4.5× more) | **beat Gopher** on almost every benchmark |

Same compute. Chinchilla was 4× smaller but trained on ~4× more data – and won.
It exposed that GPT-3-era giants (175B on ~300B tokens) were badly **undertrained**.

### Why it's true (intuition)

* **Giant model, tiny data** – huge unused capacity (data-starved).
* **Tiny model, huge data** – capacity ceiling (parameter-starved).
* The minimum-loss point is where neither bottleneck dominates – scale both together.

### Caveats

* Chinchilla optimizes **training compute only**. For heavily-served models it's often worth **over-training** a smaller model past 20 tokens/param (e.g. LLaMA) to save on inference cost.
* The "20×" is a rule of thumb – it depends on data, tokenizer, and architecture.
* Assumes you *have* enough quality tokens (the industry now faces a "data wall").


## 2. Where do "parameters" and "tokens" live?

**Two completely different things:**

| Cooking analogy | In LLMs |
| :--- | :--- |
| The material you learn from / recipes you read | **Tokens** (training data) |
| The skill stored in your brain / your memory | **Parameters** (the model) |

* **Tokens** – text chopped into pieces: `"The cat sat"` $\rightarrow$ `["The", "cat", "sat"]`. This is the *material the model reads*.
* **Parameters** – a giant pile of numbers (weights) – millions/billions of **knobs**


like `knob0 = 0.34`, `knob1 = -1.28`, ... They start **random** and get **adjusted** during training.

**Training** = tokens adjust the parameters:
1. Show tokens: `"The cat sat on the _____"`
2. Model guesses next token – wrong
3. Nudge all the knobs slightly so it's more likely correct next time
4. Repeat over trillions of tokens

The final set of knob values **is** the trained model.

Chinchilla ratio: ~20 tokens per knob $\rightarrow$ a 70B-param model wants ~1.4T tokens ($70\text{B} \times 20$).

---

## 3. An LLM is a neural network

Starting from the neuron equation:

$$Z = (W_1 \cdot X_1 + W_2 \cdot X_2 + \dots + W_n \cdot X_n) + b$$

**Correct mapping:**

| In the equation | Chinchilla term | What it is |
| :--- | :--- | :--- |
| **W** (weights) | **parameter** ☑ | the knobs |
| **b** (bias) | **parameter** ☑ | also a knob |
| **X** (input) | **token / data** ☒ | NOT a parameter |
| **Z** (output) | — | the neuron's result |

$$\text{Parameters} = \text{all the } W\text{'s} + \text{all the } b\text{'s}$$

**Why X is not a parameter:** $X$ *changes* with every new sentence (data flowing through), while $W$ and $b$ *stay fixed* after training – they are what the model learned and stores.

### Where did hidden layers & neurons go?

They didn't disappear – they're **summed into the single number N**:


more layers + more neurons $\rightarrow$ more W's and b's $\rightarrow$ more PARAMETERS (N)


**Tiny worked example** – 3 inputs $\rightarrow$ 4 hidden neurons $\rightarrow$ 1 output:
- Input $\rightarrow$ Hidden: $4 \times 3 = 12$ weights, $+ 4$ biases
- Hidden $\rightarrow$ Output: $4$ weights, $+ 1$ bias
- **Total = $12 + 4 + 4 + 1 = 21$ parameters**

So "GPT-3 has 175 billion parameters" = count all $W$ + all $b$ across every layer.
The layers and neurons are baked into that number.

An LLM is just a **very large neural network** (a **Transformer**): many layers (GPT-3 has 96), thousands of neurons wide, adding up to billions of W's and b's. Attention adds structure, but at the bottom every computation is that same neuron equation repeated billions of times.

---

## 4. "20 tokens per parameter" is a RATIO, not a wiring rule

**Common misunderstanding:** that 20 tokens are *assigned* to / *share* a specific weight. ☒

**Reality:** **Every token passes through EVERY weight.** Nothing is split or shared out.

Token 1 ("The")  – flows through ALL 70B weights $\rightarrow$ nudges them a tiny bit
Token 2 ("cat")  – flows through ALL 70B weights $\rightarrow$ nudges them a tiny bit
...
Token 1.4T       – flows through ALL 70B weights $\rightarrow$ nudges them a tiny bit

The "20" is just arithmetic on two totals:

$$\frac{\text{total training tokens}}{\text{total parameters}} = \frac{1.4\text{ trillion}}{70\text{ billion}} = 20$$


Like "4 people per household" – total-people \(\div\) total-houses, not a structural assignment.

It measures **how much data the whole network needs to be well-tuned**:
- Too few tokens for that many knobs \(\rightarrow\) undertrained (not enough practice).
- Way more tokens than needed \(\rightarrow\) wasted compute.
- Sweet spot \(\rightarrow\) 20 tokens of "practice" per knob.

**On randomness:** weights are randomly *initialized*, and tokens are fed in random *order/batches* – but neither pairs specific tokens to specific weights.

---

## 5. Sentence tokens vs. dataset tokens – two different "20s"

Example: **"The dog is barking on theft"** – 6 words, 27 characters.

Tokenization (~1.3 tokens/word, so ~7 tokens):

```text
"The"    → 1        "bark" → 1  ┐ "barking" often
"dog"    → 1        "ing"  → 1  ┘ splits into 2
"is"     → 1        "on"   → 1
                    "theft"→ 1
```

* **~7 tokens** – the sentence-level understanding is correct. ☑

**The mix-up:** Chinchilla's "20 tokens per parameter" is **NOT** about a single sentence.

| What it counts | Example |
| :--- | :--- |
| Tokens in a sentence | how one sentence is chopped up \(\mid\) ~7 tokens |
| Chinchilla's "20" | total tokens in the **ENTIRE dataset** \(\div\) total parameters \(\mid\) 1.4T \(\div\) 70B = 20 |

```text
Sentence 1: "The dog is barking on theft"   → 7 tokens
Sentence 2: "It was raining all day"        → 6 tokens
...billions more sentences (Wikipedia, books, web)...
```


...billions more sentences (Wikipedia, books, web)...

TOTAL:                           1,400,000,000,000 tokens


$$\frac{1.4\text{ trillion total tokens}}{70\text{ billion parameters}} = 20$$

**Where the 20 comes from:** not chosen – **discovered** empirically. DeepMind trained ~400 models at different sizes/data amounts, measured which gave the lowest loss for a fixed compute budget, and the winning pattern was **total tokens $\approx$ 20 $\times$ parameters**.

**Bottom line:** the sentence-token count and the dataset-level ratio are completely different scales.

---

## Key takeaways

1. An LLM **is** a neural network (a Transformer) – every computation is `$Z = \sum WX + b$`.
2. **Parameters (N)** = all the W's and b's = a summary of layers $\times$ neurons.
3. **Tokens (D)** = the text data (X) the model reads.
4. Chinchilla: for a fixed compute budget, scale N and D together $\rightarrow$ **~20 tokens per parameter**.
5. "20 tokens/param" is a **total-dataset $\div$ total-params ratio**, discovered by experiment – not a per-sentence count and not a token-to-weight wiring rule.
